# Домашнее задание: подбор гиперпараметров бустинга и интерпретация модели с SHAP

**Ожидаемое время решения:** 1–1.5 часа.

## Задание

1. Постройте модель градиентного бустинга (LightGBM) на предложенном датасете, подобрав гиперпараметры с помощью Optuna. Исследуйте результаты подбора:
   1. Постройте **parallel coordinate plot** — какие гиперпараметры, судя по графику, можно зафиксировать (не тюнить), а какие критически важны?
   2. Постройте **contour plot** для пары наиболее важных гиперпараметров — хорошо ли подобрана сетка (границы) поиска, или оптимум упирается в край диапазона?
2. Исследуйте SHAP-значения обученной модели:
   1. По **beeswarm** и/или **bar plot** определите, какие признаки являются шумовыми (не влияют на предсказание).
   2. По **dependence plot** найдите интересные попарные взаимодействия признаков.

## Датасет

Мы работаем с синтетическим датасетом **кредитного скоринга** (предсказание дефолта заёмщика).

Формат сдачи: заполненный ноутбук `homework.ipynb`, который можно запустить целиком (Run All) без ошибок.

## 0. Импорт библиотек и генерация датасета

Ниже — готовый код для генерации датасета. Его не нужно менять, просто выполните ячейки.

In [ ]:
# pip install lightgbm optuna plotly shap seaborn pandas numpy matplotlib scikit-learn

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_STATE = 42

df = pd.read_csv("credit_default_dataset.csv")
print(df.shape)
print(f"Доля дефолтов: {df['default'].mean():.3f}")
df.head()

(5000, 10)
Доля дефолтов: 0.329


,credit_score,debt_to_income,num_late_payments,employment_years,loan_amount,age,num_dependents,random_noise_1,random_noise_2,default
0,791.124188,0.392597,0,11.983732,30789.448624,40.644553,1,-0.459744,-0.302889,0
1,682.012577,0.048017,2,2.989243,16185.904729,63.539670,0,-0.101710,-0.477970,0
2,728.299039,0.220371,2,1.969566,47967.595997,39.928274,1,-0.101684,-0.676613,0
3,829.271456,0.573167,4,5.233681,33381.566076,62.364786,2,-1.101363,-0.439043,1
4,799.404639,0.451384,4,4.912667,9389.584941,32.648320,0,-0.950643,0.333650,0


### Разбиение на train/test

In [3]:
feature_cols = [c for c in df.columns if c != 'default']
X = df[feature_cols]
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)
X_train.shape, X_test.shape

((3500, 9), (1500, 9))

## 1. Подбор гиперпараметров LightGBM с помощью Optuna (4 балла)

**Задание:** напишите objective-функцию для Optuna, которая обучает `lgb.train` с подсказанными (`trial.suggest_*`) гиперпараметрами и возвращает AUC на валидационной выборке (`X_test`, `y_test`). Используйте early stopping.

Рекомендуемое пространство поиска (можно скорректировать):
- `num_leaves`: int, [10, 200]
- `learning_rate`: float, [0.01, 0.3], log scale
- `feature_fraction`: float, [0.5, 1.0]
- `bagging_fraction`: float, [0.5, 1.0]
- `bagging_freq`: int, [1, 10]
- `lambda_l1`: float, [1e-8, 10.0], log scale
- `lambda_l2`: float, [1e-8, 10.0], log scale

Запустите не менее 50 trials.

In [11]:
import optuna

train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_test, label=y_test, reference=train_data)


def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'random_state': RANDOM_STATE,
        # TODO: задайте гиперпараметры через trial.suggest_*
        'num_leaves': trial.suggest_int('nl', 10, 200),       # TODO
        'learning_rate': trial.suggest_float('lr', 0.001, 0.3, log = True),    # TODO
        'feature_fraction': trial.suggest_float('ff', 0.5, 1, step=0.1), # TODO
        'bagging_fraction': trial.suggest_float('bf', 0.5, 1, step=0.1), # TODO
        'bagging_freq': trial.suggest_int('bfreq', 1, 10),     # TODO
        'lambda_l1': trial.suggest_float('ll1', 1e-8, 10, log = True),        # TODO
        'lambda_l2': trial.suggest_float('ll2', 1e-8, 10, log = True),        # TODO
    }

    # TODO: обучите модель с early stopping и верните AUC на валидации
    model = lgb.train(params=params, train_set=train_data, valid_sets=[valid_data], callbacks=[lgb.early_stopping(stopping_rounds=5)])
    auc = roc_auc_score(y_test, model.predict(X_test))
    return auc


# TODO: создайте study и запустите optimize (n_trials >= 50)
study = optuna.create_study(direction = 'maximize')
study.optimize(objective, n_trials=100)

print("Best params:", study.best_params)
print("Best AUC:", study.best_value)

[I 2026-08-16 00:47:52,295] A new study created in memory with name: no-name-2d9ae038-afac-4b9f-9fbf-13248c75faf4
[I 2026-08-16 00:47:52,388] Trial 0 finished with value: 0.7825022006200006 and parameters: {'nl': 61, 'lr': 0.004979966228934174, 'ff': 0.9, 'bf': 1.0, 'bfreq': 5, 'll1': 1.2281840377594204, 'll2': 0.017133399389371255}. Best is trial 0 with value: 0.7825022006200006.


Training until validation scores don't improve for 5 rounds
Early stopping, best iteration is:
[5]	valid_0's auc: 0.782502
Training until validation scores don't improve for 5 rounds


[I 2026-08-16 00:47:52,584] Trial 1 finished with value: 0.8086830321622879 and parameters: {'nl': 38, 'lr': 0.04222927444633918, 'ff': 0.6, 'bf': 0.7, 'bfreq': 6, 'll1': 3.4973150770470675e-07, 'll2': 2.615115346297042e-06}. Best is trial 1 with value: 0.8086830321622879.


Early stopping, best iteration is:
[13]	valid_0's auc: 0.808683
Training until validation scores don't improve for 5 rounds


[I 2026-08-16 00:47:52,799] Trial 2 finished with value: 0.8065508982759627 and parameters: {'nl': 197, 'lr': 0.001901558707487199, 'ff': 0.5, 'bf': 0.8, 'bfreq': 8, 'll1': 0.2894170742104491, 'll2': 8.136243873030837e-05}. Best is trial 1 with value: 0.8086830321622879.


...

Training until validation scores don't improve for 5 rounds
Early stopping, best iteration is:
[24]	valid_0's auc: 0.820223
Training until validation scores don't improve for 5 rounds


[I 2026-08-16 00:48:12,101] Trial 98 finished with value: 0.8149283615099979 and parameters: {'nl': 40, 'lr': 0.08198530536241619, 'ff': 0.7, 'bf': 0.6, 'bfreq': 6, 'll1': 6.4865900001843695, 'll2': 5.28917672665003e-05}. Best is trial 77 with value: 0.8235435118470906.


Early stopping, best iteration is:
[24]	valid_0's auc: 0.814928
Training until validation scores don't improve for 5 rounds
Early stopping, best iteration is:
[21]	valid_0's auc: 0.81858


[I 2026-08-16 00:48:12,270] Trial 99 finished with value: 0.8185802828476525 and parameters: {'nl': 17, 'lr': 0.14967703874525973, 'ff': 0.7, 'bf': 0.7, 'bfreq': 5, 'll1': 3.586500611460545, 'll2': 4.7878045679074225e-06}. Best is trial 77 with value: 0.8235435118470906.


Best params: {'nl': 10, 'lr': 0.26394447749041444, 'ff': 1.0, 'bf': 0.7, 'bfreq': 7, 'll1': 2.593752601719115, 'll2': 1.615223935455776e-07}
Best AUC: 0.8235435118470906


### 1.1 Обучение финальной модели с лучшими параметрами

**Задание:** обучите LightGBM с параметрами `study.best_params` на полном train, сохраните модель в переменную `model_lgb` — она понадобится для SHAP-анализа далее.

In [12]:
best_params = study.best_params.copy()
best_params.update({'objective': 'binary', 'metric': 'auc', 'verbosity': -1, 'random_state': RANDOM_STATE})

# TODO: обучите финальную модель model_lgb с best_params
train_full = lgb.Dataset(X, label=y)
model_lgb = lgb.train(params=best_params, train_set=train_full )

# TODO: посчитайте и выведите AUC на train и test
print(f"Train:\n{roc_auc_score(y_train, model_lgb.predict(X_train))}\nTest:\n{roc_auc_score(y_test, model_lgb.predict(X_test))}")


Train:
0.9768516053851978
Test:
0.9736711175926727


### 1.2 Parallel coordinate plot

**Задание:** постройте parallel coordinate plot для `study`. Изучите график и ответьте письменно (в markdown-ячейке ниже): какие гиперпараметры, судя по графику, слабо влияют на итоговый AUC и их можно было бы зафиксировать на разумном значении по умолчанию, чтобы сократить пространство поиска?

In [13]:
from optuna.visualization import plot_parallel_coordinate

# TODO: постройте parallel coordinate plot
fig = plot_parallel_coordinate(study)
fig.show()

**Ваш ответ:** _todo_

### 1.3 Contour plot

**Задание:** выберите два гиперпараметра, которые (судя по `plot_param_importances`) сильнее всего влияют на метрику, и постройте для них contour plot. Ответьте: находится ли найденный оптимум внутри заданных границ поиска, или он «упирается» в край диапазона? Если упирается — как нужно было бы изменить границы поиска?

In [ ]:
from optuna.visualization import plot_param_importances, plot_contour

# TODO: постройте plot_param_importances(study)

# TODO: выберите два наиболее важных гиперпараметра и постройте для них plot_contour(study, params=[...])


**Ваш ответ:** _todo_

## 2. Интерпретация модели с помощью SHAP (4 балла)

**Задание:** пересчитайте SHAP-значения для обученной модели `model_lgb` на тестовой выборке с помощью `shap.TreeExplainer`.

In [ ]:
import shap

# TODO: создайте explainer
explainer = None

# TODO: посчитайте shap_values для X_test
shap_values = None

feature_names = feature_cols

### 2.1 Beeswarm / bar plot — поиск шумовых признаков

**Задание:** постройте beeswarm plot (`shap.summary_plot`) и bar plot (агрегированная важность). Определите, какие признаки являются шумовыми (SHAP value близок к нулю для всех наблюдений). Совпадает ли ваш список с тем, что вы ожидали, глядя на названия признаков?

In [ ]:
# TODO: постройте beeswarm plot
# shap.summary_plot(...)

# TODO: постройте bar plot (агрегированная важность)
# shap.summary_plot(..., plot_type='bar')


**Ваш ответ:** _todo_

### 2.2 Dependence plot — поиск попарных взаимодействий

**Задание:** постройте dependence plot для 2-3 наиболее важных признаков (по умолчанию SHAP сам подберёт цвет по признаку с наибольшим взаимодействием). Найдите признак, у которого зависимость SHAP value от значения самого признака выглядит по-разному в зависимости от цвета (значения второго признака) — это и есть взаимодействие. Дополнительно постройте dependence plot с явно заданным `interaction_index`, чтобы проверить гипотезу о конкретной паре признаков.

In [ ]:
# TODO: постройте dependence plot для наиболее важного признака (по умолчанию)
# shap.dependence_plot(..., shap_values, X_test, feature_names=feature_names)

# TODO: постройте dependence plot с явно заданным interaction_index для проверки конкретной гипотезы
# shap.dependence_plot(..., shap_values, X_test, feature_names=feature_names, interaction_index=...)


**Ваш ответ:** _todo_

## 3. Итоговые выводы (3 балла)

Кратко (в виде bullet points) резюмируйте:
- Какие гиперпараметры оказались важны, а какие можно было не тюнить?
- Была ли сетка поиска гиперпараметров задана удачно?
- Какие признаки оказались шумовыми?
- Какое взаимодействие признаков вы обнаружили и согласуется ли оно с вашей интуицией о кредитном риске?

**Ваш ответ:** _todo_